# Imports

In [1]:
import cda2
import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T
import json

from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# Connect to Spark

In [2]:
api = cda2.Api()

Set configuration parameters to better optimize queries.

In [3]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

Start Spark and specify number of cpus to use.

In [4]:
api.start_spark(n_executors=100, config=config)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


23/02/24 07:49:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
23/02/24 07:49:43 WARN DomainSocketFactory: The short-circuit local reads feature cannot be used because libhadoop cannot be loaded.
23/02/24 07:54:52 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Attempted to request executors before the AM has registered!


# Define global variables and functions

In [5]:
airports = [
    "KADW",
    "KATL",
    "KBOS",
    "KBWI",
    "KCLT",
    "KDCA",
    "KDEN",
    "KDFW",
    "KDTW",
    "KEWR",
    "KFLL",
    "KIAD",
    "KIAH",
    "KJFK",
    "KLAS",
    "KLAX",
    "KLGA",
    "KMCO",
    "KMDW",
    "KMEM",
    "KMIA",
    "KMSP",
    "KORD",
    "KPHL",
    "KPHX",
    "KSAN",
    "KSDF",
    "KSEA",
    "KSFO",
    "KSLC",
    "KTPA",
    "PANC",
    "PHNL",
]

Function to convert Unix timestamp (milliseconds from 1970) to YYYYMMDD string.

In [6]:
year0 = "2019"
year1 = str(int(year0) + 1)

In [7]:
dates = {"start_date": year0 + "-01-01", "end_date": year1 +"-01-01"}

# Retrieve SIDs and STARs for the airports of interest

<li>only get SID and STAR procedure types. (procedure_type = "APPROACH" is ignored)</li>
<li>create a unique <i>grouping</i> column.</li>
</br>
this will retrieve multiple versions of each fix, one for each update cycle in the year.

In [8]:
df_procedures = (
    api.dataframe("ArincTransition", **dates, metadata=True)
    .select(
        "dtpp_procedure_name",
        "procedure_name",
        "procedure_type",
        "transition_name",
        "transition_type",
        F.col("airport.icao_region").alias("icao_region"),
        F.col("airport.name").alias("airport"),
        F.explode("legs").alias("leg"),
        F.col("leg.path_terminator.identification.name").alias("fix_name"),
        F.col("leg.sequence_number").alias("seq"),
        F.col("leg.path_terminator.latitude"),
        F.col("leg.path_terminator.longitude"),
        F.col("leg.path_terminator.magnetic_variation.modeled").alias("magnetic_variation"),
        F.col("leg.speed_limit.descriptor").alias("speed_description"),
        F.col("leg.speed_limit.limit").alias("speed_limit"),
        F.col("leg.speed_limit.altitude").alias("speed_altitude"),
        F.col("leg.altitude_limits.description").alias("altitude_description"),
        F.col("leg.altitude_limits.value1").alias("altitude_value1"),
        F.col("leg.altitude_limits.value2").alias("altitude_value2"),
        F.col("metadata.effective_end_date").alias("end_date"),
    )
    .withColumn("grouping", F.concat("procedure_name", F.lit("_"), "transition_name", F.lit("_"), "seq"))
    .filter(F.col("transition_name").isNotNull())
    .filter(F.col("fix_name").isNotNull())
    .filter(F.col("procedure_type").isin("SID", "STAR"))

     .filter(F.col("icao_region").startswith("K") | F.col("icao_region").startswith("PA") | F.col("icao_region").startswith("PH"))

#    comment the following out so we get procedures for all US airports
#    .filter(F.col("airport").isin(airports)) 


    .drop("leg")   
    .orderBy("grouping")
)
#df_procedures.show()

## keep only the newest update

In [9]:
window = Window.partitionBy("grouping").orderBy(col("end_date").desc())

df_procedures_newest = (df_procedures
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row", "end_date")
    .orderBy("grouping")
)
#df_procedures_newest.show(50)

## output procedures (SIDs and STARs) partioned by airport

In [10]:
(
    df_procedures_newest
        .repartition("airport")
        .select(
            "dtpp_procedure_name",
            "procedure_name",
            "procedure_type",
            "transition_name",
            "transition_type",
            "fix_name",
            "seq",
            "latitude",
            "longitude",
            "magnetic_variation",
            "airport",
            "speed_description",
            "speed_limit",
            "speed_altitude",
            "altitude_description",
            "altitude_value1",
            "altitude_value2"
        )
        .orderBy("grouping")
        .write.option("header",True).partitionBy(["airport"])
        .csv("CRAFT/" + year0 + "/procedures", compression="None", mode="overwrite")
)

## output all north american procedure fixes
<li>add a column that concats all the procedures that use the fix</li>
<li>add a column that concats the procedure types that use the fix</li>

In [11]:
(
    df_procedures_newest.groupBy("fix_name", "icao_region", "latitude", "longitude", "magnetic_variation")
        .agg(F.concat_ws(":", F.collect_set("procedure_name")).alias("procedures_using_fix"), F.concat_ws(":", F.collect_set("procedure_type")).alias("procedure_types_using_fix"))
        .orderBy("fix_name")
        .write.option("header", True)
        .csv("CRAFT/" + year0 + "/procedures", compression="None", mode="append")
)